In [ ]:
import os
import warnings
from pathlib import Path
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')

ROOT = Path('..') if Path('.').name == 'notebooks' else Path('.')
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

def _warn(message):
    print(f'WARNING: {message}')

def load_wfp_prices(path):
    if not os.path.exists(path):
        _warn(f'WFP price file not found: {path}')
        return None

    df = pd.read_csv(path)
    print('WFP columns:', list(df.columns))
    print(df.head(5))

    df = df[df['adm1_name'].astype(str).str.contains('Jawa Timur', case=False, na=False)]
    df = df[df['commodity'].astype(str).str.contains('Rice', case=False, na=False)]
    df = df[['date', 'market', 'commodity', 'price', 'unit']].copy()
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df = df.dropna(subset=['date', 'price'])
    df = df.set_index('date').resample('MS')['price'].mean().reset_index()
    df = df.rename(columns={'date': 'ds', 'price': 'y_wfp'})

    if not df.empty:
        assert df['ds'].min() < pd.Timestamp('2015-01-01'), 'WFP ds min should be before 2015'
        assert df['ds'].max() > pd.Timestamp('2022-01-01'), 'WFP ds max should be after 2022'

    return df

def load_pihps_malang(path):
    if not os.path.exists(path):
        _warn(f'PIHPS Malang file not found: {path}')
        return None

    df = pd.read_excel(path, skiprows=2)
    print('PIHPS columns:', list(df.columns))

    cols = [str(c) for c in df.columns]
    date_cols = [c for c in cols if 'tanggal' in c.lower() or 'date' in c.lower() or 'tgl' in c.lower() or 'bulan' in c.lower()]
    price_cols = [c for c in cols if 'beras medium i' in c.lower() or 'beras medium' in c.lower()]

    if not date_cols:
        _warn('Could not find an exact date column in PIHPS file; using first date-like column.')
        date_cols = [c for c in cols if 'bulan' in c.lower() or 'tahun' in c.lower() or 'tgl' in c.lower()]
    if not price_cols:
        _warn('Could not find Beras Medium column exactly; selecting first numeric column as price.')
        numeric = df.select_dtypes(include=[np.number]).columns.tolist()
        price_cols = [numeric[0]] if numeric else []

    if not date_cols or not price_cols:
        _warn('PIHPS Malang file does not contain identifiable date or price columns.')
        return None

    date_col = date_cols[0]
    price_col = price_cols[0]
    df[date_col] = pd.to_datetime(df[date_col], dayfirst=True, errors='coerce')
    df = df[[date_col, price_col]].rename(columns={date_col: 'ds', price_col: 'y_pihps'})
    df = df.dropna(subset=['ds', 'y_pihps'])
    df = df.set_index('ds').resample('MS')['y_pihps'].mean().reset_index()
    return df

def load_bps_production(path):
    if not os.path.exists(path):
        _warn(f'BPS production file not found: {path}')
        return None

    df = pd.read_excel(path, skiprows=3)
    print('BPS columns:', list(df.columns))

    cols = [str(c) for c in df.columns]
    date_cols = [c for c in cols if 'tahun' in c.lower() or 'month' in c.lower() or 'bulan' in c.lower()]
    region_cols = [c for c in cols if 'jawa timur' in c.lower()]

    if not date_cols or not region_cols:
        _warn('Could not identify BPS columns for date or Jawa Timur production.')
        return None

    date_col = date_cols[0]
    region_col = region_cols[0]
    df = df[[date_col, region_col]].rename(columns={date_col: 'ds', region_col: 'production_gkg'})
    df['ds'] = pd.to_datetime(df['ds'], errors='coerce')
    df['production_gkg'] = pd.to_numeric(df['production_gkg'], errors='coerce')
    df = df.dropna(subset=['ds', 'production_gkg'])
    df['month'] = df['ds'].dt.month
    df['production_dev_pct'] = df.groupby('month')['production_gkg'].transform(lambda x: (x - x.mean()) / x.mean())
    return df[['ds', 'production_gkg', 'production_dev_pct']].copy()

def load_bmkg_rainfall(path):
    if not os.path.exists(path):
        _warn(f'BMKG rainfall file not found: {path}')
        return None

    df = pd.read_csv(path)
    print('BMKG columns:', list(df.columns))

    cols = [str(c) for c in df.columns]
    date_cols = [c for c in cols if 'date' in c.lower() or 'tanggal' in c.lower()]
    rain_cols = [c for c in cols if 'rain' in c.lower() or 'curah' in c.lower() or 'mm' in c.lower()]

    if not date_cols:
        _warn('Could not identify BMKG date column.')
        return None
    if not rain_cols:
        numeric = df.select_dtypes(include=[np.number]).columns.tolist()
        rain_cols = [numeric[0]] if numeric else []

    if not rain_cols:
        _warn('Could not identify BMKG rainfall column.')
        return None

    date_col = date_cols[0]
    rain_col = rain_cols[0]
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    df = df.dropna(subset=[date_col, rain_col])
    df = df.set_index(date_col).resample('MS')[rain_col].sum().reset_index()
    df = df.rename(columns={date_col: 'ds', rain_col: 'rainfall_mm'})
    df['month'] = df['ds'].dt.month
    df['rainfall_dev_pct'] = df.groupby('month')['rainfall_mm'].transform(lambda x: (x - x.mean()) / x.mean())
    return df[['ds', 'rainfall_mm', 'rainfall_dev_pct']].copy()

def build_harvest_features(date_series):
    ds = pd.to_datetime(date_series)
    months = ds.dt.month
    harvest_window = months.isin([3, 4, 5, 7, 8, 9]).astype(int)
    lean_season = months.isin([10, 11, 12, 1, 2]).astype(int)

    def days_to_next_march(date):
        year = date.year if date.month < 3 else date.year + 1
        target = pd.Timestamp(year=year, month=3, day=1)
        return max(0, (target - date).days)

    harvest_proximity_days = [0 if hw == 1 else days_to_next_march(d) for d, hw in zip(ds, harvest_window)]
    return pd.DataFrame({
        'ds': ds,
        'harvest_window': harvest_window,
        'lean_season': lean_season,
        'harvest_proximity_days': harvest_proximity_days,
    })

def select_price_series(df_wfp, df_pihps):
    if df_wfp is not None and df_pihps is not None:
        df = df_wfp.merge(df_pihps, on='ds', how='inner')
        df['y'] = df[['y_wfp', 'y_pihps']].mean(axis=1)
        print('Using merged WFP + PIHPS price series.')
    elif df_wfp is not None:
        df = df_wfp.rename(columns={'y_wfp': 'y'})[['ds', 'y']]
        _warn('Only WFP price series available.')
    elif df_pihps is not None:
        df = df_pihps.rename(columns={'y_pihps': 'y'})[['ds', 'y']]
        _warn('Only PIHPS price series available.')
    else:
        raise ValueError('No price series available from WFP or PIHPS.')

    assert len(df) >= 48, 'Final price series must contain at least 48 rows.'
    return df[['ds', 'y']].copy()

def build_master_df(price_df, bps_df, bmkg_df, harvest_df):
    df = price_df.copy()
    if bps_df is not None:
        df = df.merge(bps_df, on='ds', how='left')
    if bmkg_df is not None:
        df = df.merge(bmkg_df, on='ds', how='left')
    df = df.merge(harvest_df, on='ds', how='left')
    prod_missing = df['production_dev_pct'].isna().sum() if 'production_dev_pct' in df.columns else len(df)
    rain_missing = df['rainfall_dev_pct'].isna().sum() if 'rainfall_dev_pct' in df.columns else len(df)
    df['production_dev_pct'] = df.get('production_dev_pct', pd.Series(0.0, index=df.index)).fillna(0.0)
    df['rainfall_dev_pct'] = df.get('rainfall_dev_pct', pd.Series(0.0, index=df.index)).fillna(0.0)
    print(f'Filled {prod_missing} missing production_dev_pct rows with 0.')
    print(f'Filled {rain_missing} missing rainfall_dev_pct rows with 0.')
    df = df.dropna(subset=['y'])
    print(f'Final master_df shape: {df.shape}')
    print(f'Date range: {df["ds"].min().strftime("%Y-%m")} to {df["ds"].max().strftime("%Y-%m")}')
    df.to_csv(PROCESSED_DIR / 'master_df.csv', index=False)
    return df

wfp = load_wfp_prices(RAW_DIR / 'wfp_food_prices_idn.csv')
pihps = load_pihps_malang(RAW_DIR / 'pihps_malang_beras.xlsx')
bps = load_bps_production(RAW_DIR / 'bps_produksi_padi_bulanan.xlsx')
bmkg = load_bmkg_rainfall(RAW_DIR / 'bmkg_rainfall_jatim.csv')
price_df = select_price_series(wfp, pihps)
harvest_df = build_harvest_features(price_df['ds'])
master_df = build_master_df(price_df, bps, bmkg, harvest_df)
missing_y = master_df['y'].isna().sum()
assert missing_y == 0, 'Master data contains missing y values.'
print('=== DATA QUALITY REPORT ===')
source_name = 'WFP + PIHPS' if (wfp is not None and pihps is not None) else ('WFP' if wfp is not None else 'PIHPS')
print(f'Price series:      {source_name}')
print(f'Date range:        {master_df["ds"].min().strftime("%Y-%m")} to {master_df["ds"].max().strftime("%Y-%m")}')
print(f'Total rows:        {len(master_df)}')
print(f'Missing y:         {missing_y} (must be 0 — crash if not)')
print(f'Missing prod_dev:  {prod_missing} rows filled with 0')
print(f'Missing rain_dev:  {rain_missing} rows filled with 0')
print('Feature summary:')
print(f'  harvest_window=1:  {(master_df["harvest_window"] == 1).sum()} rows ({(master_df["harvest_window"] == 1).mean() * 100:.1f}%)')
print(f'  lean_season=1:     {(master_df["lean_season"] == 1).sum()} rows ({(master_df["lean_season"] == 1).mean() * 100:.1f}%)')
print('  price_mom_3m:      computed in next notebook')
print(f'  production_dev_pct range: [{master_df["production_dev_pct"].min():.3f}, {master_df["production_dev_pct"].max():.3f}]')
print(f'Columns: {master_df.columns.tolist()}')
print('===========================')
train_df = master_df[master_df['ds'] < '2024-01-01']
holdout_df = master_df[master_df['ds'] >= '2024-01-01']
train_df.to_csv(PROCESSED_DIR / 'train_df.csv', index=False)
holdout_df.to_csv(PROCESSED_DIR / 'holdout_df.csv', index=False)
print(f'Train rows:   {len(train_df)}')
print(f'Holdout rows: {len(holdout_df)}')
assert len(train_df) >= 48, 'Training set must have at least 48 rows'
assert len(holdout_df) >= 6, 'Holdout set must have at least 6 rows'